# ⚡ Z-Fooocus
**Multi-Model Studio** — Z-Image Turbo · FLUX.2-klein · Qwen-Image-Edit

Generate · Img2Img · Inpaint with model switching

In [ ]:
# ── Step 1: Install & Download Models ─────────────────────
import os

# ComfyUI
if not os.path.exists('/content/ComfyUI'):
    !git clone https://github.com/comfyanonymous/ComfyUI.git /content/ComfyUI
    %cd /content/ComfyUI
    !pip install -q -r requirements.txt

# ComfyUI-GGUF custom nodes (for Qwen-Image-Edit GGUF)
if not os.path.exists('/content/ComfyUI/custom_nodes/ComfyUI-GGUF'):
    !git clone https://github.com/city96/ComfyUI-GGUF.git /content/ComfyUI/custom_nodes/ComfyUI-GGUF
    !pip install -q -r /content/ComfyUI/custom_nodes/ComfyUI-GGUF/requirements.txt

# Extra deps
!pip install -q rembg[gpu] onnxruntime-gpu huggingface_hub

# Model dirs
UNET = '/content/ComfyUI/models/unet'
CLIP = '/content/ComfyUI/models/clip'
VAE  = '/content/ComfyUI/models/vae'
os.makedirs(UNET, exist_ok=True)
os.makedirs(CLIP, exist_ok=True)
os.makedirs(VAE,  exist_ok=True)

from huggingface_hub import hf_hub_download

def dl(repo, filename, local_dir, force_filename=None):
    dest = os.path.join(local_dir, force_filename or os.path.basename(filename))
    if os.path.exists(dest):
        print(f'✅ Already exists: {os.path.basename(dest)}')
        return
    print(f'⏳ Downloading {os.path.basename(filename)}...')
    hf_hub_download(repo_id=repo, filename=filename, local_dir=local_dir)
    # Rename if needed
    downloaded = os.path.join(local_dir, filename)
    if force_filename and os.path.exists(downloaded) and downloaded != dest:
        os.rename(downloaded, dest)
    print(f'✅ Done: {os.path.basename(dest)}')

# ── Z-Image Turbo FP8 ─────────────────────────────────────
dl('Tongyi-MAI/Z-Image-Turbo',
   'z-image-turbo-fp8-e4m3fn.safetensors', UNET)

# Qwen 3.4B CLIP (for Z-Image Turbo + FLUX)
dl('Comfy-Org/Qwen_3_4B_Lumina2_CLIP',
   'qwen_3_4b.safetensors', CLIP)

# VAE for Z-Image Turbo + FLUX
dl('Comfy-Org/flux1-dev',
   'ae.safetensors', VAE)

# ── FLUX.2-klein FP8 (~10GB) ──────────────────────────────
dl('black-forest-labs/FLUX.2-klein-9b-kv-fp8',
   'flux-2-klein-9b-kv-fp8.safetensors', UNET)

# ── Qwen-Image-Edit-2511 GGUF Q4_K_M ─────────────────────
dl('unsloth/Qwen-Image-Edit-2511-GGUF',
   'qwen-image-edit-2511-Q4_K_M.gguf', UNET)

# Qwen2.5-VL CLIP GGUF (text encoder for Qwen-Image-Edit)
dl('unsloth/Qwen2.5-VL-7B-Instruct-GGUF',
   'Qwen2.5-VL-7B-Instruct-UD-Q4_K_XL.gguf', CLIP)

# Qwen-Image VAE
dl('Qwen/Qwen-Image-Edit-2511',
   'vae/diffusion_pytorch_model.safetensors', VAE, 'qwen_image_vae.safetensors')

print('\n🎉 All models ready!')
print('   Z-Image Turbo FP8     ✅')
print('   FLUX.2-klein FP8      ✅')
print('   Qwen-Image-Edit GGUF  ✅')

In [ ]:
# ── Step 2: Launch Z-Fooocus ──────────────────────────────
import os
os.chdir('/content')
REPO = 'https://github.com/MuntasirMalek/Z-Fooocus.git'
if os.path.exists('/content/Z-Fooocus'):
    %cd /content/Z-Fooocus
    !git pull --ff-only || true
else:
    !git clone {REPO}
    %cd /content/Z-Fooocus
!python app.py